# hallu_hierarchy — fixed runner (CLAUDE.md §5)
Same 4 cells every run.
- **CLI runs** (`scripts/kaggle_run.sh`): the script fills in `RUN_REQUEST` in a temporary copy of this notebook (stages, config, commit). Nothing here is edited by hand.
- **Browser fallback** (Save & Run All): leave `RUN_REQUEST = None` and edit only the `MANUAL` line in cell 2.
Secrets: `HF_TOKEN` (optional), `GITHUB_TOKEN` (only if the repo is private). Internet on.
Read the result in `REPORT.md` (last block of the log, `REPORT START` … `REPORT END`).

In [ ]:
RUN_REQUEST = None  # filled in by scripts/kaggle_run.sh for CLI runs; keep None for browser runs

# 1) Setup — code goes to /tmp so /kaggle/working holds only REPORT.md, DASHBOARD.png, REVIEW.csv, outputs/
import os, subprocess
from kaggle_secrets import UserSecretsClient

def secret(name):
    try:
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

if hf := secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = hf
gh = secret('GITHUB_TOKEN')
REPO = 'github.com/harshith20/Long_Range_MASE.git'
CODE = '/tmp/hallu_repo'
subprocess.run(['rm', '-rf', CODE], check=True)
subprocess.run(['git', 'clone', '-q', f'https://{gh}@{REPO}' if gh else f'https://{REPO}', CODE], check=True)
os.chdir(CODE)
if RUN_REQUEST:  # run exactly the commit the script checked was pushed
    subprocess.run(['git', 'checkout', '-q', RUN_REQUEST['commit']], check=True)
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
!git log -1 --oneline
!pip install -q -r hallu_hierarchy/requirements.txt
!python -m pytest -q -p no:warnings -p no:cacheprovider hallu_hierarchy/tests 2>&1 | tail -n 3

In [ ]:
# 2) Run — browser runs: edit only MANUAL. CLI runs: RUN_REQUEST (cell 1) decides.
MANUAL = '--config configs/qwen3b_popqa_dev.yaml --stages correctness'
ARGS = f"--config {RUN_REQUEST['config']} --stages {RUN_REQUEST['stages']}" if RUN_REQUEST else MANUAL
print('[notebook] python -m hallu_hierarchy.run', ARGS)
!python -m hallu_hierarchy.run {ARGS}

In [ ]:
# 3) Report — printed last so it is the end of the log
from pathlib import Path
from IPython.display import Image, display
W = Path('/kaggle/working')
if (W / 'DASHBOARD.png').exists():
    display(Image(str(W / 'DASHBOARD.png')))
print((W / 'REPORT.md').read_text() if (W / 'REPORT.md').exists()
      else 'REPORT.md missing: the run stopped before writing it. See the error above.')

In [ ]:
# 4) Save — /kaggle/working is kept as this version's Output. To reuse it in a later stage:
# Output tab -> New Dataset, then attach that dataset as an input (it is found automatically).
import shutil
for junk in ('.virtual_documents', '__pycache__'):
    shutil.rmtree(f'/kaggle/working/{junk}', ignore_errors=True)